# Lab 2 — The Lie Detector
**Session 2 · Prompt engineering + your first eval harness · TCE**

You'll need: your **10-question file from Lab 1 Part E**.
First: **File → Save a copy in Drive**.

In [ ]:
# Cell 1 — setup (same as Lab 1)
%pip install -q -U google-genai
from getpass import getpass
from google import genai
from google.genai import types
import logging; logging.getLogger("google_genai").setLevel(logging.ERROR)   # hide harmless SDK notices (e.g. "non-text parts: thought_signature")
import time

MOCK = False   # ← set True only if the instructor says the live API is unavailable

client = genai.Client(api_key="mock" if MOCK else getpass("Gemini API key: "))
MODEL = "gemini-flash-lite-latest"  # lite: minimal thinking by default (near-zero thinking tokens on simple prompts), so the free tier goes further (Aug 2026 → Gemini 3.5 Flash Lite). 429 = rate limit: the helper below waits and retries; 503 = high demand: wait and re-run.

_MOCK_ANSWERS = [   # (keyword, canned reply) — offline path only. One fact below is deliberately wrong: the eval should catch it.
    ("pass or fail",     "PASS"),
    ("reply only a or b", "A"),                                   # a judge with a position bias — Stretch 3 will catch it
    ("roja",             "The music for Roja (1992) was composed by A. R. Rahman — it was his debut film score."),
    ("tce",              "Thiagarajar College of Engineering (TCE), Madurai, was founded in 1957 by Karumuttu Thiagarajan Chettiar. It is an autonomous institution affiliated to Anna University, known for its engineering programmes and its 100-acre campus on the Madurai–Theni road."),
    ("2011",             "India won the 2011 Cricket World Cup, beating Sri Lanka in the final at the Wankhede Stadium, Mumbai, under captain Virat Kohli."),   # ← wrong on purpose (it was Dhoni)
]
_MOCK_DEFAULT = "I am not sure."

def ask(prompt, temperature=None):
    if MOCK:
        p = prompt.lower()
        return "[MOCK] " + next((a for k, a in _MOCK_ANSWERS if k in p), _MOCK_DEFAULT)
    config = types.GenerateContentConfig(temperature=temperature) if temperature is not None else None
    for attempt in range(4):
        try:
            # blocked/empty responses come back as None — treat as an empty answer, it scores as a miss
            return client.models.generate_content(model=MODEL, contents=prompt, config=config).text or ""
        except Exception as e:
            if "429" in str(e) and attempt < 3:
                print("rate limited, waiting..."); time.sleep(20*(attempt+1))
            else: raise
print("ready ✓  If any cell's import fails: Runtime -> Restart session, then re-run from this cell."
      + ("  (MOCK mode — canned answers)" if MOCK else ""))

import re, textwrap
def wrap(text, width=80):
    """Word-wrap a reply so it fits the screen. Keeps the model's own line breaks, bullets and code blocks."""
    out, in_code = [], False
    for line in str(text).splitlines():
        if line.lstrip().startswith("```"):
            in_code = not in_code
        if in_code or line.lstrip().startswith("```") or not line.strip():
            out.append(line); continue
        m = re.match(r"\s*(?:[-*•]|\d+[.)])\s+", line)          # bullet / numbered item → hang-indent
        hang = " " * len(m.group(0)) if m else line[:len(line) - len(line.lstrip())]
        out.append(textwrap.fill(line, width, subsequent_indent=hang,
                                 break_long_words=False, break_on_hyphens=False))
    return "\n".join(out)

## Part A — The prompt makeover (5 iterations)

Below is a deliberately terrible prompt. Improve it **five times**, one upgrade per run:
**v1** task (length+subject) → **v2** role+audience → **v3** context (real facts) → **v4** format+negative instructions → **v5** constraint ("only stated facts").

Run, read, then edit `PROMPT` and run again. Document each step in the table below.

The ten-question evaluation later is a learning instrument, not a reliable estimate of production accuracy. Keep evaluation questions separate from prompt examples, and keep them fixed while comparing prompts.

In [ ]:
# Cell 2 — edit PROMPT, run, repeat (keep old versions in comments!)
PROMPT = "write about tce"   # v0 — terrible on purpose

print(wrap(ask(PROMPT)))

### Document your makeover (edit this cell)

| v | What I changed | Why the output got better |
|---|---|---|
| 1 | | |
| 2 | | |
| 3 | | |
| 4 | | |
| 5 | | |

### ✓ Checkpoint 1 — five documented iterations.

---
## Part B — Your test set → your first eval

Fill `my_tests` from your Lab 1 Part E file. Keep `expected` SHORT — the key fact only (a name, a number), not a full sentence. The scorer checks whether your expected string appears inside the model's answer (after normalization).

In [ ]:
# Cell 3 — your 10 questions (3 examples shown — replace with YOURS)
my_tests = [
    {"q": "Who composed the music for the film Roja?",                     "expected": "rahman"},
    {"q": "In which year was TCE Madurai founded?",                         "expected": "1957"},
    {"q": "Who captained India in the 2011 Cricket World Cup final?",       "expected": "dhoni"},
    # ... add your 7+ more ...
    # No list ready? sample-inputs/sample-10-questions.txt (lab folder) has ten with their key facts.
]
print(len(my_tests), "questions loaded")

In [ ]:
# Cell 4 — the eval harness
import re, textwrap

def norm(s):
    """lowercase + drop punctuation, so 'A. R. Rahman!' and 'a r rahman' compare equal"""
    return re.sub(r"[^a-z0-9 ]", "", s.lower())

def short(ans, n=240):
    """the answer on one line, cut to n characters — for printing"""
    s = " ".join(str(ans).split())
    return s if len(s) <= n else s[:n] + " …"

def grade(template, tests):
    """Ask every question with this prompt template. Returns one result per question."""
    if not tests:
        raise ValueError("Add at least one labelled test before running the evaluation")
    results = []
    for t in tests:
        ans = ask(template.format(q=t["q"]), temperature=0.0)
        results.append({"q": t["q"], "expected": t["expected"], "answer": ans,
                        "ok": norm(t["expected"]) in norm(ans)})      # the check: is the key fact in the answer?
    return results

def show(r, label="got:"):
    """print one graded answer: ✓/✗, the question, what we expected, what came back"""
    print(("✓" if r["ok"] else "✗"), r["q"])
    print("   expected:", r["expected"])
    print(textwrap.fill(short(r["answer"]), 80, initial_indent=f"   {label:<10}", subsequent_indent=" " * 13))
    print()

def run_eval(template, tests, verbose=True):
    """Grade ONE prompt. verbose=True prints the evidence for every question."""
    results = grade(template, tests)
    if verbose:
        for r in results: show(r)
    hits = sum(r["ok"] for r in results)
    score = hits / len(results)
    print(f"SCORE: {hits}/{len(results)} = {score:.0%}   (n={len(results)} — evidence about these questions only)")
    return score

baseline = run_eval("Answer this question: {q}", my_tests)

### Read every ✗ before moving on
For each failure, decide: **model wrong** (hallucination — the interesting case), **scorer too strict** (fix your `expected` string), or **question ambiguous** (fix the question). This diagnosis IS the skill.

**Scored 9 or 10 out of 10?** Your set is too easy to measure anything — swap in 5 obscure questions (deep cuts, not headlines) before Part C.

### ✓ Checkpoint 2 — eval ran on your 10 questions; failures diagnosed.

---
## Part C — Prompt A vs Prompt B, settled with numbers

In [ ]:
# Cell 5 — design a better template, then fight (on the first 5 questions while you iterate)
PROMPT_A = "Answer this question: {q}"

# Only {q} may appear in braces — a literal { } in your template will crash .format(). Ask for 'JSON with keys name, degree, year' in words instead.
PROMPT_B = (
    "You are a careful expert. Answer the question below.\n"
    "Rules: be direct, give the specific fact asked for, "
    "and if you are not sure, say 'I am not sure' instead of guessing.\n\n"
    "Question: {q}\nAnswer:"
)   # ← edit me — beat A by more!

def compare(prompt_a, prompt_b, tests):
    """Run both prompts on the SAME questions and print a side-by-side scorecard."""
    ra, rb = grade(prompt_a, tests), grade(prompt_b, tests)
    n, W = len(tests), 56
    print(f"    {'question':<{W}}  A  B")
    print("    " + "─" * (W + 6))
    for i, (a, b) in enumerate(zip(ra, rb), 1):
        q = a["q"] if len(a["q"]) <= W else a["q"][:W - 1] + "…"
        note = "  ← B fixed it" if b["ok"] and not a["ok"] else "  ← B broke it" if a["ok"] and not b["ok"] else ""
        print(f"{i:>2}. {q:<{W}}  {'✓' if a['ok'] else '✗'}  {'✓' if b['ok'] else '✗'}{note}")
    sa, sb = sum(r["ok"] for r in ra), sum(r["ok"] for r in rb)
    print("    " + "─" * (W + 6))
    print(f"    {'correct answers':<{W}} {sa:>2} {sb:>2}   (out of {n})")

    missed = [(i, a, b) for i, (a, b) in enumerate(zip(ra, rb), 1) if not (a["ok"] and b["ok"])]
    if missed:
        print("\nEvery question where A or B missed — read both answers:\n")
        for i, a, b in missed:
            print(f"{i}. {a['q']}   (expected: {a['expected']})")
            for tag, r in (("A", a), ("B", b)):
                print(textwrap.fill(short(r["answer"], 200), 80,
                                    initial_indent=f"   {tag} {'✓' if r['ok'] else '✗'}  ", subsequent_indent=" " * 8))
            print()

    if sb > sa:   verdict = f"B WINS by {sb - sa} question(s): {sb}/{n} vs {sa}/{n}."
    elif sa > sb: verdict = f"A WINS by {sa - sb}: your new prompt made things worse. Read the '← B broke it' rows."
    elif sa == n: verdict = (f"TIE at {sa}/{n} — both prompts got EVERY question right, so these questions cannot "
                             "tell the prompts apart. Add harder questions (local facts, exact numbers, obscure names) until one fails.")
    else:         verdict = f"TIE at {sa}/{n} — same score. Read the ✗ answers above and change PROMPT_B to fix them."
    print("\n" + textwrap.fill("VERDICT: " + verdict, 80, subsequent_indent=" " * 9))
    if n < 10 and sa != sb:
        print(f"         n={n}: a {abs(sa - sb)}-question gap on {n} questions is a hint, not proof — confirm with Cell 5b.")
    return sa / n, sb / n

DEV = my_tests[:5]     # iterate here: 5 calls per prompt per run, not 10 — you have another lab today
print(f"Prompt A vs Prompt B on the first {len(DEV)} questions  (✓ = key fact found in the answer)\n")
score_a, score_b = compare(PROMPT_A, PROMPT_B, DEV)

In [ ]:
# Cell 5b — CONFIRMATION RUN on all 10 (run ONCE, when PROMPT_B is final — 20 calls)
# A win on 5 questions is a hint, not a result. Confirm on the full set before you claim it.
print(f"CONFIRMATION: Prompt A vs Prompt B on ALL {len(my_tests)} questions\n")
score_a_all, score_b_all = compare(PROMPT_A, PROMPT_B, my_tests)

def leader(a, b): return "B ahead" if b > a else "A ahead" if a > b else "tie"
print(f"\nfirst {len(DEV)} questions : A {score_a:.0%}  vs  B {score_b:.0%}  → {leader(score_a, score_b)}")
print(f"all {len(my_tests)} questions   : A {score_a_all:.0%}  vs  B {score_b_all:.0%}  → {leader(score_a_all, score_b_all)}")
print("Did the early leader survive the full run?",
      "YES — report both numbers WITH n." if leader(score_a, score_b) == leader(score_a_all, score_b_all)
      else "NO — the 5-question result was noise. That is exactly why we confirm.")

### ✓ Checkpoint 3 — show me A vs B numbers + the single most interesting failure you found.

---
## Stretch goals

In [ ]:
# Stretch 1 — variance: is your score stable?
# Same prompt, same questions, asked 3 times. If the score moves, a 1-question "win" in Cell 5 might be luck.
RUNS = 3
runs = [grade(PROMPT_B, my_tests) for _ in range(RUNS)]
scores = [sum(r["ok"] for r in res) / len(res) for res in runs]

for i, res in enumerate(runs, 1):
    print(f"run {i}: {sum(r['ok'] for r in res)}/{len(res)} = {scores[i-1]:.0%}")

print(f"\nper question, across the {RUNS} runs:")
for j, t in enumerate(my_tests):
    marks = "".join("✓" if runs[i][j]["ok"] else "✗" for i in range(RUNS))
    wording = len({" ".join(runs[i][j]["answer"].split()) for i in range(RUNS)})
    status = "stable" if len(set(marks)) == 1 else "FLIPS between runs"
    print(f"  {marks}  {status:<18}  {'same words' if wording == 1 else f'{wording} wordings':<10}  {t['q'][:44]}")

spread = max(scores) - min(scores)
print(f"\nscore range: {min(scores):.0%} – {max(scores):.0%}  →",
      "stable: one run is enough at temperature 0." if spread == 0 else
      f"it moved by {spread:.0%} with NOTHING changed. A prompt 'win' smaller than that is noise.")
# We used temperature=0.0 in the harness — try changing it in grade() and watch stability change.

In [ ]:
# Stretch 2 — LLM-as-judge (a model grades a model)
def judge_score(question, expected, answer):
    verdict = ask(
        f"Question: {question}\nExpected key fact: {expected}\nStudent answer: {answer}\n"
        "Does the student answer contain the expected fact (paraphrase ok)? Reply only PASS or FAIL.",
        temperature=0.0)
    return "PASS" in verdict.upper()

print("Two graders, same answers: the keyword check (Cell 4) vs a model acting as judge\n")
agree = 0
for t in my_tests[:3]:                               # 3 questions × (1 answer + 1 judgement) = 6 calls
    ans = ask(PROMPT_B.format(q=t["q"]), temperature=0.0)
    kw = norm(t["expected"]) in norm(ans)
    jd = judge_score(t["q"], t["expected"], ans)
    agree += (kw == jd)
    print(t["q"])
    print("   expected:", t["expected"])
    print(textwrap.fill(short(ans, 200), 80, initial_indent="   answer:   ", subsequent_indent=" " * 13))
    print(f"   keyword check: {'PASS' if kw else 'FAIL'}    judge: {'PASS' if jd else 'FAIL'}")
    print("   →", "they agree" if kw == jd else "they DISAGREE — read the answer yourself: which grader is right?")
    print()
print(f"graders agreed on {agree}/3.")
# Where might the judge itself be wrong? (too lenient, prefers long answers, agrees with itself...)

In [ ]:
# Stretch 3 — position bias: a PAIRWISE judge, run both ways
# A judge that compares two answers can prefer whichever comes first. Swap the order and see how often it changes its mind.
def judge_pair(question, a, b):
    """Returns 'A' or 'B' — which of the two answers the judge prefers."""
    v = ask(f"Question: {question}\n\nAnswer A: {a}\n\nAnswer B: {b}\n\n"
            "Which answer is better — more correct, more specific? Reply only A or B.", temperature=0.0)
    v = v.removeprefix("[MOCK] ").strip().upper()
    return "B" if v.startswith("B") else "A"

flips = 0
for t in my_tests[:3]:                                   # 3 questions × (2 answers + 2 judgements) = 12 calls
    a, b = ask(PROMPT_A.format(q=t["q"])), ask(PROMPT_B.format(q=t["q"]))
    first  = judge_pair(t["q"], a, b)                    # A shown first
    second = judge_pair(t["q"], b, a)                    # B shown first — so 'A' here means B won
    second = "B" if second == "A" else "A"               # map back to the real labels
    flips += first != second
    print(t["q"])
    print(f"   A shown first → judge picks {first}")
    print(f"   B shown first → judge picks {second}")
    print("   →", "same pick both ways: the CONTENT decided" if first == second
          else "it CHANGED ITS MIND: the ORDER decided, not the content")
    print()
print(f"position-bias flips: {flips}/3 — every flip is a verdict decided by ORDER, not by which answer is better")

## Wrap
You now own the loop: **prompt → eval → read failures → fix → re-run.** Keep this notebook — the same harness grades your capstone in Session 6.

Before you close: **File → Save a copy in Drive** again (your test set only exists in this notebook), and paste `my_tests` into a text file too.

**Short break. Session 3: AI gets eyes and ears — have a photo or two on your phone.**